In [ ]:
""""
Assignment 4, Task 1:

This file contains code to do the following initial
steps for Assignment 4:
- Load Quotes Data & Executions Data
- Feature Engineering
- Train scikit-learn based regression models

Classmates Cited: Annie Reynolds
"""

In [ ]:
#Imports and Global Variable Defintions:
import pandas as pd
import numpy as np

import joblib
from joblib import dump, load

import sklearn
from sklearn import ensemble, impute, pipeline, preprocessing, tree
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

BUY = 1

In [ ]:
#Quotes Data Loading & Cleaning:
QUOTES_INPUT = pd.read_csv('quotes_2025-09-10_small.csv.gz',
                    dtype = {
                        'ticker': 'category'
                    }
                    )
QUOTES_COLS = ['ticker', 'bid_price', 'ask_price', 'ask_size', 'bid_exchange', 'ask_exchange', 'sip_timestamp']

#Convert to Proper Date Time:
QUOTES_INPUT['sip_timestamp'] = pd.to_datetime(QUOTES_INPUT['sip_timestamp'], 
                                               errors = 'coerce', infer_datetime_format=True)
QUOTES_INPUT = QUOTES_INPUT.dropna(subset=['sip_timestamp'])

#Drop bid_exchange and ask_exchange:
QUOTES_INPUT = QUOTES_INPUT.drop(columns=['bid_exchange', 'ask_exchange'])

#Convert to Proper Numeric Types:
QUOTES_INPUT['ask_price'] = pd.to_numeric(QUOTES_INPUT['ask_price'], errors='coerce')
QUOTES_INPUT['bid_price'] = pd.to_numeric(QUOTES_INPUT['bid_price'], errors='coerce')

#Filter for Trading Hours Only:
QUOTES_INPUT = QUOTES_INPUT[(QUOTES_INPUT['sip_timestamp'] >= '2025-09-10 09:30:00') 
                            & (QUOTES_INPUT['sip_timestamp'] <= '2025-09-10 16:00:00')]

QUOTES_INPUT = QUOTES_INPUT.dropna()
QUOTES_INPUT = QUOTES_INPUT.rename(columns={'ticker': 'symbol'})
QUOTES_INPUT = QUOTES_INPUT.sort_values(['sip_timestamp', 'symbol']).reset_index(drop=True)

/var/folders/1_/gb2cs27x3nq4pp7f6whksw7h0000gn/T/ipykernel_12247/29739547.py:12: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  QUOTES_INPUT['sip_timestamp'] = pd.to_datetime(QUOTES_INPUT['sip_timestamp'], errors = 'coerce', infer_datetime_format=True)


In [55]:
#Executions Data Loading & Cleaning:
EXECUTIONS_INPUT = pd.read_csv('execs_from_fix.csv').dropna()

#Convert to Proper Data Types:
EXECUTIONS_INPUT['order_time'] = pd.to_datetime(EXECUTIONS_INPUT['order_time'])
EXECUTIONS_INPUT['execution_time'] = pd.to_datetime(EXECUTIONS_INPUT['execution_time'])
EXECUTIONS_INPUT['limit_price'] = pd.to_numeric(EXECUTIONS_INPUT['limit_price'], errors='coerce')
EXECUTIONS_INPUT['execution_price'] = pd.to_numeric(EXECUTIONS_INPUT['execution_price'], errors='coerce')
EXECUTIONS_INPUT['order_qty'] = pd.to_numeric(EXECUTIONS_INPUT['order_qty'], errors='coerce')
EXECUTIONS_INPUT['order_id'] = EXECUTIONS_INPUT['order_id'].astype('category')
EXECUTIONS_INPUT['symbol'] = EXECUTIONS_INPUT['symbol'].astype('category')
EXECUTIONS_INPUT['exchange'] = EXECUTIONS_INPUT['exchange'].astype('category')
EXECUTIONS_INPUT['side'] = EXECUTIONS_INPUT['side'].apply(lambda x: 'B' if x == 1 else 'S')
EXECUTIONS_INPUT['side'] = EXECUTIONS_INPUT['side'].astype('category')

#Filter for Market Hours:
EXECUTIONS_INPUT = EXECUTIONS_INPUT[(EXECUTIONS_INPUT['order_time'] >= '2025-09-10 09:30:00') 
                                    & (EXECUTIONS_INPUT['order_time'] <= '2025-09-10 16:00:00')]

EXECUTIONS_INPUT.head()

,order_id,order_time,execution_time,symbol,side,order_qty,limit_price,execution_price,exchange
7844,ID86335,2025-09-10 09:30:00.997,2025-09-10 09:30:01.099,NIO,B,50,5.66,5.66,ID1516
7847,ID86355,2025-09-10 09:30:05.000,2025-09-10 09:30:05.103,MWYN,B,225,2.65,2.65,ID1516
7849,ID86363,2025-09-10 09:30:07.162,2025-09-10 09:30:07.266,STKE,B,18,8.30,8.00,ID1516
7851,ID86382,2025-09-10 09:30:10.105,2025-09-10 09:30:10.210,CUPR,S,89,2.93,3.03,ID1516
7852,ID86399,2025-09-10 09:30:13.755,2025-09-10 09:30:13.857,CUPR,S,60,3.00,3.02,ID1516


In [ ]:
#Combine Executions and Quotes to Calculate Price Improvement:
def calculate_price_improvement(execution_row: pd.Series) -> float:
    """Calculate the price improvement for a given order"""
    if execution_row['side'] == BUY:
        return max(0, execution_row['limit_price'] - execution_row['execution_price'])
    else: #Sell Order
        return max(0, execution_row['execution_price'] - execution_row['limit_price'])
    
#Prepare Symbol Columns to Match for Merge:
EXECUTIONS_INPUT['symbol'] = EXECUTIONS_INPUT['symbol'].astype(str)
QUOTES_INPUT['symbol'] = QUOTES_INPUT['symbol'].astype(str)

#Sort Inputs for Merge:
EXECUTIONS_INPUT = EXECUTIONS_INPUT.sort_values([ 'order_time', 'symbol'])
QUOTES_INPUT = QUOTES_INPUT.sort_values(['sip_timestamp', 'symbol'])

executions_and_quotes = pd.merge_asof(
    EXECUTIONS_INPUT,
    QUOTES_INPUT,
    left_on='order_time',
    right_on='sip_timestamp',
    by='symbol',
    direction='backward' #helps find the earliest prior quote
)

executions_and_quotes['price_improvement'] = executions_and_quotes.apply(calculate_price_improvement, axis=1)
executions_and_quotes.head()

,order_id,order_time,execution_time,symbol,side,order_qty,limit_price,execution_price,exchange,bid_price,ask_price,bid_size,ask_size,sip_timestamp,price_improvement
0,ID86335,2025-09-10 09:30:00.997,2025-09-10 09:30:01.099,NIO,B,50,5.66,5.66,ID1516,NaN,NaN,NaN,NaN,NaT,0.0
1,ID86355,2025-09-10 09:30:05.000,2025-09-10 09:30:05.103,MWYN,B,225,2.65,2.65,ID1516,2.61,2.65,5.0,2.0,2025-09-10 09:30:04.181625568,0.0
2,ID86359,2025-09-10 09:30:05.962,2025-09-10 09:33:37.670,CDTG,B,1000,1.30,1.30,ID1516,1.30,1.31,45.0,19.0,2025-09-10 09:30:05.736571594,0.0
3,ID86363,2025-09-10 09:30:07.162,2025-09-10 09:30:07.266,STKE,B,18,8.30,8.00,ID1516,NaN,NaN,NaN,NaN,NaT,0.0
4,ID86369,2025-09-10 09:30:07.228,2025-09-10 09:37:43.187,AEHL,B,100,12.00,12.00,ID1516,13.33,13.64,1.0,1.0,2025-09-10 09:30:06.628618340,0.0


In [ ]:
#Group into Exchange GroupBy Items:
exchanges = executions_and_quotes.groupby('exchange')

def find_viable_exchanges(exchanges):
    """Find the exchanges that have enough data to train on

    Input:
        exchanges (GroupBy): all exchanges found in executions_and_quotes
    Output:
        viable_exchanges (Dict: str, DataFrame): dictionary containing exchange
            names and their corresponding executions_and_quotes data for training
    """
    viable_exchanges = {}
    for exchange_name, exchange_data in exchanges:
        #Ensure there's enough data for training, after dropping NA values:
        exchange_data = exchange_data.dropna(subset=['side', 'order_qty', 'limit_price', 
                                                     'bid_price', 'ask_price', 'bid_size', 
                                                     'ask_size', 'price_improvement'])
        if len(exchange_data) < 5:
            print(f'Not enough data to train on {exchange_name}')
        else:
            viable_exchanges[exchange_name] = exchange_data
    return viable_exchanges


def train_singular_exchange(exchange_data, exchange_name, save_model=True):
    """
    For a single exchange, train a Gradient Boosted Trees model to predict price improvement.
    Inputs:
        exchange_data: DataFrame containing executions and quotes for a single exchange
        exchange_name: Name of the exchange
        save_model: indicates whether or not to save the trained model to disk
    Outputs:
        model_metrics: metrics for the trained model (rmse, R^2)
    """
    #Drop NA Values:
    exchange_data = exchange_data.dropna(subset=['side', 'order_qty', 'limit_price', 
                                                 'bid_price', 'ask_price', 'bid_size', 
                                                 'ask_size', 'price_improvement'])

    #Convert Side to Numeric:
    exchange_data['side'] = exchange_data['side'].apply(lambda x: 1 if x == 'B' else 2)

    #Define Inputs & Target from Executions DataFrame:
    inputs = exchange_data[['side', 'order_qty', 'limit_price', 
                            'bid_price', 'ask_price', 'bid_size', 'ask_size']]
    target = exchange_data['price_improvement']

    #Split Into Training & Testing Sets:
    train_inputs, test_inputs, train_target, test_target = train_test_split(
        inputs,
        target,
        test_size=0.2,
        random_state=42
    )

    #Hyperparameter Tuning Attempts:
    gbt_tuning_params = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    }

    rf_tuning_params = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 6, 10]
    }

    #Define the model: start with default hyperparameters
    
    #GBT APPROACH
    #base_GBT_model = sklearn.ensemble.GradientBoostingRegressor(
    #    random_state=42
    #)
    #model = sklearn.model_selection.GridSearchCV(
    #    estimator=base_GBT_model,
    #    param_grid=gbt_tuning_params,
    #    scoring='r2',
    #    cv=3,
    #    n_jobs=-1,
    #    verbose=1
    #)

    #RANDOM FOREST APPROACH:
    base_RF_model = sklearn.ensemble.RandomForestRegressor(random_state=42)
    model = sklearn.model_selection.GridSearchCV(
        estimator=base_RF_model,
        param_grid=rf_tuning_params,
        scoring='r2',
        cv=3,
        n_jobs=-1, #apply hint in assignment writeup
        verbose=1
    )

    model.fit(train_inputs, train_target)

    #Identify Best Model from Hyperparameter Tuning:
    best_model = model.best_estimator_

    #Evaluate Best Model on Test Set:
    predictions = best_model.predict(test_inputs)
    rmse = np.sqrt(mean_squared_error(test_target, predictions))
    r2 = r2_score(test_target, predictions)

    #Save Trained Model to Disk:
    if save_model:
        filename = f'gbt_price_improvement_model_{exchange_name}.joblib'
        joblib.dump(best_model, filename)
        print(f'Saved Model for {exchange_name} to {filename}')

    model_metrics = {'exchange': exchange_name, 
                     'model': best_model,
                     'rmse': rmse,
                     'r2': r2,
                     'best_hyperparameters' : model.best_params_}

    return model_metrics

In [ ]:
#Identify Viable Exchanges for Training:
viable_exchanges = find_viable_exchanges(exchanges)
print(viable_exchanges.keys())

#Train Model for Each Viable Exchange:
for exchange_name, exchange_data in viable_exchanges.items():
    print(f'Training using data from Exchange ID: {exchange_name}')
    train_results = train_singular_exchange(exchange_data, exchange_name)
    print(train_results)


Not enough data to train on ID211917
Not enough data to train on ID245333
Not enough data to train on ID282763
Not enough data to train on ID295386
Not enough data to train on ID412967
Not enough data to train on ID422100
Not enough data to train on ID524810
dict_keys(['ID1516', 'ID29608'])
Training using data from Exchange ID: ID1516
Fitting 3 folds for each of 27 candidates, totalling 81 fits
Saved Model for ID1516 to gbt_price_improvement_model_ID1516.joblib
{'exchange': 'ID1516', 'model': RandomForestRegressor(max_depth=15, n_estimators=300, random_state=42), 'rmse': np.float64(0.06868991539320328), 'r2': 0.13178731339876382, 'best_hyperparameters': {'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 300}}
Training using data from Exchange ID: ID29608
Fitting 3 folds for each of 27 candidates, totalling 81 fits
Saved Model for ID29608 to gbt_price_improvement_model_ID29608.joblib
{'exchange': 'ID29608', 'model': RandomForestRegressor(max_depth=10, n_estimators=300, random_sta